# Diabetes Care Information Assistant (Roll No.: 1024160015)

In [ ]:
import nltk
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

taxonomy = {
    "Diabetes Mellitus": {
        "Type 1 Diabetes": ["juvenile diabetes", "insulin-dependent"],
        "Type 2 Diabetes": ["adult-onset", "insulin resistance"]
    },
    "Diagnostic Metrics": {
        "Blood Glucose": ["fasting sugar", "postprandial sugar"],
        "HbA1c": ["glycated hemoglobin", "three-month average"]
    },
    "Management": {
        "Pharmacotherapy": ["insulin therapy", "oral hypoglycemics"],
        "Lifestyle Modification": ["diet control", "physical exercise"]
    },
    "Complications": {
        "Acute Complications": ["diabetic ketoacidosis", "hypoglycemia"],
        "Chronic Complications": ["neuropathy", "retinopathy"]
    }
}

concepts = ["Diabetes", "Insulin", "Glucose", "Diet", "Metformin", "HbA1c", "Hypoglycemia", "Ketoacidosis"]
relationships = [
    ("Diabetes", "characterized_by", "Glucose"),
    ("Insulin", "regulates", "Glucose"),
    ("Metformin", "treats", "Diabetes"),
    ("Diet", "manages", "Glucose"),
    ("Hypoglycemia", "is_low_level_of", "Glucose"),
    ("Ketoacidosis", "is_complication_of", "Diabetes"),
    ("HbA1c", "measures", "Glucose"),
    ("Pharmacotherapy", "includes", "Metformin"),
    ("Type 1 Diabetes", "requires", "Insulin"),
    ("Lifestyle Modification", "incorporates", "Diet")
]

data = [
    # Greetings
    ("hello", "greetings"),
    ("hi there", "greetings"),
    ("hey assistant", "greetings"),
    ("good morning", "greetings"),
    ("anyone there", "greetings"),
    ("yo", "greetings"),
    ("whats up", "greetings"),
    ("howdy", "greetings"),

    # Symptoms
    ("what are the signs of diabetes", "symptoms"),
    ("common symptoms of high sugar level", "symptoms"),
    ("frequent urination and feeling extremely thirsty", "symptoms"),
    ("blurry vision and constant fatigue", "symptoms"),
    ("how do I know if I have diabetes", "symptoms"),
    ("tell me warning signs of type 2 diabetes", "symptoms"),
    ("unexplained weight loss and hunger", "symptoms"),
    ("dry mouth and itchy skin symptoms", "symptoms"),

    # Diet
    ("what should a diabetic patient eat", "diet"),
    ("foods to avoid with high sugar levels", "diet"),
    ("can I eat sweet fruits if I have diabetes", "diet"),
    ("best diet chart for type 2 diabetes", "diet"),
    ("is low carb food good for sugar patients", "diet"),
    ("should I stop eating white rice completely", "diet"),
    ("diabetic friendly snacks and meals", "diet"),
    ("suggest some good food choices for blood sugar control", "diet"),

    # Medication
    ("tell me about metformin dosage", "medication"),
    ("how does insulin work for type 1 diabetes", "medication"),
    ("side effects of common diabetes tablets", "medication"),
    ("when is the best time to take my sugar medication", "medication"),
    ("do I need insulin injections daily", "medication"),
    ("prescribed medicine options for blood glucose control", "medication"),
    ("what happens if I skip my diabetes pill", "medication"),
    ("information about diabetes drugs and therapy", "medication"),

    # Emergency
    ("help my sugar dropped to 40 and I am passing out", "emergency"),
    ("unconscious diabetic patient with rapid breathing", "emergency"),
    ("severe diabetic ketoacidosis symptoms emergency", "emergency"),
    ("extreme confusion and shaking with high insulin dose", "emergency"),
    ("emergency chest pain and diabetic coma risk", "emergency"),
    ("very high ketones and vomiting medical help needed", "emergency"),
    ("blood sugar is 500 and I am losing consciousness", "emergency"),
    ("severe hyperosmolar hyperglycemic state assistance", "emergency")
]

df = pd.DataFrame(data, columns=["text", "intent"])

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    cleaned = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and word not in string.punctuation]
    return " ".join(cleaned)

df["processed"] = df["text"].apply(preprocess)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["processed"])
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf_eval = LogisticRegression(C=1.0)
clf_eval.fit(X_train, y_train)
y_pred = clf_eval.predict(X_test)

print("Classification Evaluation\n")
print(classification_report(y_test, y_pred, zero_division=0))
print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred))

model = LogisticRegression(C=2.0)
model.fit(X, y)

kb = {
    "greetings": "Hello! I am your Diabetes Care Information Assistant. How can I assist you today?",
    "symptoms": "Common diabetes symptoms include frequent urination, excessive thirst, extreme fatigue, blurry vision, and unexplained weight loss. Please consult a doctor for a professional medical diagnosis.",
    "diet": "A healthy diabetes diet focuses on complex carbs (whole grains, oats), lean proteins, non-starchy vegetables, and healthy fats. Avoid refined sugars, sweetened beverages, and heavy carbs.",
    "medication": "Diabetes medications include oral therapies like Metformin to lower glucose production, and Insulin therapy to help control sugar level. Follow your prescription closely."
}

def get_response(user_query):
    emergency_keywords = ["passing out", "unconscious", "coma", "severe", "chest pain", "vomiting", "emergency", "shaking", "dying", "hospital"]
    if any(kw in user_query.lower() for kw in emergency_keywords):
        return "EMERGENCY: Severe symptoms detected. Please immediately contact emergency services or go to the nearest hospital!"

    healthcare_keywords = ["sugar", "diabetes", "diabetic", "insulin", "metformin", "symptom", "diet", "eat", "food", "pill", "dose", "glucose", "carb"]
    greetings_keywords = ["hi", "hello", "hey", "yo", "morning", "howdy", "sup"]

    is_health = any(kw in user_query.lower() for kw in healthcare_keywords)
    is_greet = any(kw in user_query.lower().split() for kw in greetings_keywords)

    if not (is_health or is_greet):
         return "UNSUPPORTED: I can only help with diabetes-related topics. Ask about symptoms, diet, or medications."

    cleaned = preprocess(user_query)
    if not cleaned.strip():
         return "Can you rephrase your question?"

    vect = vectorizer.transform([cleaned])
    probs = model.predict_proba(vect)[0]
    max_idx = np.argmax(probs)
    confidence = probs[max_idx]
    predicted_intent = model.classes_[max_idx]

    if confidence < 0.28:
         return "LOW CONFIDENCE: I'm not sure. Try asking more specifically about symptoms, diet, or medications."

    if predicted_intent == "emergency":
         return "EMERGENCY: Severe symptoms detected. Please call an ambulance or rush to the nearest emergency ward immediately."

    return kb.get(predicted_intent, "How can I help with your diabetes today?")

print("\nTesting Diabetes Care Information Assistant\n")

test_queries = [
    "Hello assistant!",
    "What foods should I avoid with high sugar?",
    "Can you tell me how to build a web app?",
    "My blood sugar level is normal but sometimes I feel weird.",
    "Help me! I am having severe chest pain and dizzy fits."
]

for q in test_queries:
    print(f"User: {q}")
    print(f"Assistant: {get_response(q)}\n")

Classification Evaluation

              precision    recall  f1-score   support

        diet       0.00      0.00      0.00         2
   emergency       0.00      0.00      0.00         2
   greetings       0.67      1.00      0.80         2
  medication       0.50      0.50      0.50         2
    symptoms       0.00      0.00      0.00         2

    accuracy                           0.30        10
   macro avg       0.23      0.30      0.26        10
weighted avg       0.23      0.30      0.26        10


Confusion Matrix

[[0 1 0 0 1]
 [2 0 0 0 0]
 [0 0 2 0 0]
 [1 0 0 1 0]
 [0 0 1 1 0]]

Testing Diabetes Care Information Assistant

User: Hello assistant!
Assistant: Hello! I am your Diabetes Care Information Assistant. How can I assist you today?

User: What foods should I avoid with high sugar?
Assistant: A healthy diabetes diet focuses on complex carbs (whole grains, oats), lean proteins, non-starchy vegetables, and healthy fats. Avoid refined sugars, sweetened beverages, and h

In [ ]:
print("Interactive Diabetes Assistant Chat")
print("Type your query below. Type 'exit' or 'quit' to end the chat.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit"]:
        print("Assistant: Take care! Goodbye.")
        break
    if not user_input.strip():
        continue
    response = get_response(user_input)
    print(f"Assistant: {response}\n")

Interactive Diabetes Assistant Chat
Type your query below. Type 'exit' or 'quit' to end the chat.

You: Hey there!
Assistant: Hello! I am your Diabetes Care Information Assistant. How can I assist you today?

You: What are the signs of having diabetes?
Assistant: Common diabetes symptoms include frequent urination, excessive thirst, extreme fatigue, blurry vision, and unexplained weight loss. Please consult a doctor for a professional medical diagnosis.

You: What should a diabetic patient eat?
Assistant: A healthy diabetes diet focuses on complex carbs (whole grains, oats), lean proteins, non-starchy vegetables, and healthy fats. Avoid refined sugars, sweetened beverages, and heavy carbs.

You: How does metformin help control glucose?
Assistant: Diabetes medications include oral therapies like Metformin to lower glucose production, and Insulin therapy to help control sugar level. Follow your prescription closely.

You: Which food is helpful for managing sugar peaks?
Assistant: A healt